# Scalability evaluation

This notebook evaluates the scalability of the SPARQL query templates using the QLever Wikidata endpoint.

The evaluation uses query templates stored in the `query templates/` folder. The templates are tested with four input scales: 1, 10, 100, and 1000 entities. The input entities are sampled from entities that have at least one final reviewed change-related entity-property pair. Each query is executed five times, with a 60-second interval between repeated runs. For each run, the notebook records the number of input entities, matched entities, returned rows, runtime, execution status, and possible errors.

The scalability results reported in the thesis are based on the saved raw log file:
`qlever_all_query_log_20260614_025034.csv`

This saved log is used to reproduce the thesis summary table and figures. Because QLever is an online endpoint and Wikidata is continuously updated, rerunning the queries may produce slightly different runtimes, matched entities, or returned rows. Therefore, the rerun cells are useful for checking the current behaviour of the query templates, while the thesis reproduction cells are used to reproduce the results reported in the thesis.

## Optional: run QLever queries again

This cell below executes all query templates against the QLever Wikidata endpoint and creates a new timestamped raw log file.

The output of this cell may differ from the thesis results because QLever and Wikidata may change over time. The generated raw log can be used to inspect the current behaviour of the query templates.

In [ ]:
import requests
import time
import pandas as pd
import re
from pathlib import Path
from io import StringIO
from datetime import datetime


QLEVER_URL = "https://qlever.dev/api/wikidata"

template_folder = Path("query_templates")
result_folder = Path("qlever_results")
result_folder.mkdir(exist_ok=True)

query_files = sorted(template_folder.glob("*.sparql"))

N_RUNS = 5
SLEEP_SECONDS = 60
REQUEST_TIMEOUT = 60


log_file = f"qlever_all_query_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

print("Number of query files:", len(query_files))
print("Log file:", log_file)

# Check query file names before running
for f in query_files:
    print(f.name)

all_logs = []


# Run all queries

for query_index, query_file in enumerate(query_files, start=1):
    query = query_file.read_text(encoding="utf-8")

    for run_id in range(1, N_RUNS + 1):
        print(f"\nQuery {query_index}/{len(query_files)}: {query_file.name}, Run {run_id}/{N_RUNS}")

        start_time = time.perf_counter()
        response = None

        try:
            response = requests.post(
                QLEVER_URL,
                data={"query": query},
                headers={"Accept": "text/csv"},
                timeout=REQUEST_TIMEOUT
            )

            runtime = time.perf_counter() - start_time
            input_entities = len(set(re.findall(r"wd:Q\d+", query)))
            http_status_code = response.status_code

            if response.status_code == 200:
                result_df = pd.read_csv(StringIO(response.text))

                matched_entities = result_df["entity"].nunique() if "entity" in result_df.columns else None
                result_rows = len(result_df)

                status = "success"
                error_message = ""

                result_file = result_folder / f"{query_file.stem}_run{run_id}.csv"
                result_df.to_csv(result_file, index=False)

            else:
                matched_entities = None
                result_rows = None
                status = "failed"
                error_message = response.text[:500]
                result_file = None

        except requests.exceptions.Timeout:
            runtime = REQUEST_TIMEOUT
            input_entities = len(set(re.findall(r"wd:Q\d+", query)))
            http_status_code = None
            matched_entities = None
            result_rows = None
            status = "timeout"
            error_message = f"Timeout after {REQUEST_TIMEOUT} seconds"
            result_file = None

        except Exception as e:
            runtime = None
            input_entities = len(set(re.findall(r"wd:Q\d+", query)))
            http_status_code = response.status_code if response is not None else None
            matched_entities = None
            result_rows = None
            status = "error"
            error_message = str(e)
            result_file = None

        # Save one run's log
        one_log = {
            "query_file": query_file.name,
            "query_index": query_index,
            "run_id": run_id,
            "input_entities": input_entities,
            "matched_entities": matched_entities,
            "result_rows": result_rows,
            "runtime_seconds": runtime,
            "status": status,
            "http_status_code": http_status_code,
            "error_message": error_message,
            "result_file": str(result_file) if result_file else None
        }

        all_logs.append(one_log)

        # Save log after every run
        log_df = pd.DataFrame(all_logs)
        log_df.to_csv(log_file, index=False)

        print("Status:", status)
        print("HTTP status code:", http_status_code)
        print("Input entities:", input_entities)
        print("Matched entities:", matched_entities)
        print("Result rows:", result_rows)
        print("Runtime seconds:", runtime)

        # Wait 60 seconds, except after the final run
        is_last_query = query_index == len(query_files)
        is_last_run = run_id == N_RUNS

        if not (is_last_query and is_last_run):
            print(f"Waiting {SLEEP_SECONDS} seconds...")
            time.sleep(SLEEP_SECONDS)

print("\nAll done!")
print("Log file saved as:", log_file)

Number of query files: 72
Log file: qlever_all_query_log_20260709_165841.csv
business_scale1000_CQ10_reference.sparql
business_scale1000_CQ1_recorded_value.sparql
business_scale1000_CQ2_current_value_qualifier.sparql
business_scale1000_CQ2_current_value_qualifier_rank.sparql
business_scale1000_CQ3_value_at_2024.sparql
business_scale1000_CQ4_lifecycle_event.sparql
business_scale1000_CQ5_CQ8_temporal_info.sparql
business_scale1000_CQ9_sequence_property.sparql
business_scale1000_CQ9_sequence_qualifier.sparql
business_scale100_CQ10_reference.sparql
business_scale100_CQ1_recorded_value.sparql
business_scale100_CQ2_current_value_qualifier.sparql
business_scale100_CQ2_current_value_qualifier_rank.sparql
business_scale100_CQ3_value_at_2024.sparql
business_scale100_CQ4_lifecycle_event.sparql
business_scale100_CQ5_CQ8_temporal_info.sparql
business_scale100_CQ9_sequence_property.sparql
business_scale100_CQ9_sequence_qualifier.sparql
business_scale10_CQ10_reference.sparql
business_scale10_CQ1_reco

In [ ]:
# 1. Read the raw log generated by the previous cell
input_file = log_file
df = pd.read_csv(input_file)

# 2. Parse domain, scale, cq from query_file
def parse_query_file(query_file):
    # example: business_scale10_CQ1_recorded_value.sparql
    name = str(query_file).replace(".sparql", "")
    
    domain_match = re.match(r"(business|politician)_", name)
    scale_match = re.search(r"scale(\d+)", name)
    cq_match = re.search(r"(CQ\d+)", name)
    
    domain = domain_match.group(1) if domain_match else None
    scale = int(scale_match.group(1)) if scale_match else None
    cq = cq_match.group(1) if cq_match else None
    
    # template name without domain and scale
    template = name
    template = re.sub(r"^(business|politician)_scale\d+_", "", template)
    
    return pd.Series({
        "domain": domain,
        "scale": scale,
        "cq": cq,
        "template": template
    })

parsed = df["query_file"].apply(parse_query_file)
df = pd.concat([parsed, df], axis=1)

# 3. Only use successful runs to calculate runtime
success_df = df[df["status"] == "success"].copy()

# 4. Pivot 5 runtimes horizontally：run_1, run_2, ..., run_5
runtime_wide = success_df.pivot_table(
    index=["domain", "cq", "template", "scale", "query_file"],
    columns="run_id",
    values="runtime_seconds",
    aggfunc="first"
).reset_index()

runtime_wide = runtime_wide.rename(columns={
    1: "run_1_runtime_seconds",
    2: "run_2_runtime_seconds",
    3: "run_3_runtime_seconds",
    4: "run_4_runtime_seconds",
    5: "run_5_runtime_seconds",
})

# 5. Summary basic info for each query template
summary_base = (
    df.groupby(["domain", "cq", "template", "scale", "query_file"], dropna=False)
    .agg(
        total_runs=("run_id", "count"),
        success_runs=("status", lambda x: (x == "success").sum()),
        timeout_runs=("status", lambda x: (x == "timeout").sum()),
        failed_runs=("status", lambda x: ((x != "success") & (x != "timeout")).sum()),
        input_entities=("input_entities", "max"),
        matched_entities=("matched_entities", "first"),
        returned_results=("result_rows", "first"),
    )
    .reset_index()
)

# 6. Summary runtime：average, min, max, range, standard deviation
runtime_summary = (
    success_df.groupby(["domain", "cq", "template", "scale", "query_file"], dropna=False)
    .agg(
        average_runtime_seconds=("runtime_seconds", "mean"),
        min_runtime_seconds=("runtime_seconds", "min"),
        max_runtime_seconds=("runtime_seconds", "max"),
        std_runtime_seconds=("runtime_seconds", "std"),
    )
    .reset_index()
)

summary = summary_base.merge(
    runtime_summary,
    on=["domain", "cq", "template", "scale", "query_file"],
    how="left"
)

summary["runtime_range_seconds"] = (
    summary["max_runtime_seconds"] - summary["min_runtime_seconds"]
)

summary["success_rate"] = summary["success_runs"] / summary["total_runs"]

# 7. Merge the 5 specific runtimes
summary = summary.merge(
    runtime_wide,
    on=["domain", "cq", "template", "scale", "query_file"],
    how="left"
)

# 8. Mark prominent cases
def mark_prominent(row):
    reasons = []

    if row["timeout_runs"] > 0:
        reasons.append("has timeout")

    if row["failed_runs"] > 0:
        reasons.append("has failed run")

    if pd.notna(row["runtime_range_seconds"]) and row["runtime_range_seconds"] > 0.5:
        reasons.append("large runtime variation")

    if pd.notna(row["average_runtime_seconds"]) and row["average_runtime_seconds"] > 1:
        reasons.append("slow query")

    if len(reasons) == 0:
        return "normal"
    else:
        return "; ".join(reasons)

summary["prominent_note"] = summary.apply(mark_prominent, axis=1)

# 9. Sort by domain, CQ, scale
# CQ1, scale 1, 10, 100, 1000; then CQ2, scale 1, 10, 100, 1000
cq_order = {f"CQ{i}": i for i in range(1, 11)}
scale_order = {1: 1, 10: 2, 100: 3, 1000: 4}
domain_order = {"business": 1, "politician": 2}

summary["domain_order"] = summary["domain"].map(domain_order)
summary["cq_order"] = summary["cq"].map(cq_order)
summary["scale_order"] = summary["scale"].map(scale_order)

summary = summary.sort_values(
    by=["domain_order", "cq_order", "template", "scale_order"]
)

summary = summary.drop(columns=["domain_order", "cq_order", "scale_order"])

# 10. Adjust column order 
columns_order = [
    "domain",
    "cq",
    "template",
    "scale",
    "query_file",
    "total_runs",
    "success_runs",
    "timeout_runs",
    "failed_runs",
    "success_rate",
    "input_entities",
    "matched_entities",
    "returned_results",
    "average_runtime_seconds",
    "min_runtime_seconds",
    "max_runtime_seconds",
    "runtime_range_seconds",
    "std_runtime_seconds",
    "prominent_note",
    "run_1_runtime_seconds",
    "run_2_runtime_seconds",
    "run_3_runtime_seconds",
    "run_4_runtime_seconds",
    "run_5_runtime_seconds",
]

summary = summary[[col for col in columns_order if col in summary.columns]]

# 11. Write to the second sheet
output_xlsx = "query_results_with_summary_current.xlsx"
output_csv = "query_results_with_summary_current.csv"

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="all_runs_raw_log", index=False)
    summary.to_excel(writer, sheet_name="average_summary", index=False)

summary.to_csv(output_csv, index=False)

print(f"Input raw log: {input_file}")
print(f"Saved to {output_xlsx}")
print(f"Saved to {output_csv}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import math


file_path = "query_results_with_summary_current.xlsx"
df = pd.read_excel(file_path, sheet_name="average_summary")

# 2. Keep needed data
# Remove CQ4 because it shares the same retrieval structure as CQ1
# but mention CQ4 together with CQ1 in the subplot title
df = df[df["cq"] != "CQ4"].copy()

# Keep only business and politician
df = df[df["domain"].isin(["business", "politician"])].copy()


# 3. Define template order and labels

template_order = [
    "CQ1_recorded_value",
    "CQ2_current_value_qualifier",
    "CQ2_current_value_qualifier_rank",
    "CQ3_value_at_2024",
    "CQ5_CQ8_temporal_info",
    "CQ9_sequence_qualifier",
    "CQ9_sequence_property",
    "CQ10_reference",
]

template_label_map = {
    "CQ1_recorded_value": "CQ1, CQ4\nRecorded value retrieval",
    "CQ2_current_value_qualifier": "CQ2\nCurrent value retrieval\nwith qualifier",
    "CQ2_current_value_qualifier_rank": "CQ2\nCurrent value retrieval\nwith qualifier and rank",
    "CQ3_value_at_2024": "CQ3\nValue retrieval at a given time",
    "CQ5_CQ8_temporal_info": "CQ5–CQ8\nTemporal information retrieval",
    "CQ9_sequence_qualifier": "CQ9\nSequence relation retrieval\nwith qualifier",
    "CQ9_sequence_property": "CQ9\nSequence relation retrieval\nwith property",
    "CQ10_reference": "CQ10\nReference retrieval",
}

scale_order = [1, 10, 100, 1000]

df = df[df["template"].isin(template_order)].copy()
df["template_order"] = df["template"].map({t: i for i, t in enumerate(template_order)})
df["scale_order"] = df["scale"].map({s: i for i, s in enumerate(scale_order)})
df = df.sort_values(["domain", "template_order", "scale_order"])


# 4. Plot function

def plot_small_plots_with_range(domain_name, y_tick_step=0.2):
    domain_df = df[df["domain"] == domain_name].copy()

    # ---- determine one common y-axis range for all 8 subplots ----
    domain_ymax = domain_df["max_runtime_seconds"].max()

    # round up to nearest tick step
    upper_limit = math.ceil(domain_ymax / y_tick_step) * y_tick_step

    # give a little extra space for annotations
    upper_limit = upper_limit + y_tick_step

    fig, axes = plt.subplots(
        2, 4,
        figsize=(19, 8.1),
        sharex=True,
        sharey=True   # all subplots use the same y scale
    )

    axes = axes.flatten()

    x_positions = np.arange(len(scale_order))
    x_labels = [str(s) for s in scale_order]

    for ax, template in zip(axes, template_order):
        temp_df = domain_df[domain_df["template"] == template].copy()
        temp_df = temp_df.sort_values("scale_order")

        avg_values = []
        min_values = []
        max_values = []
        point_labels = []

        for scale in scale_order:
            row = temp_df[temp_df["scale"] == scale]

            if row.empty:
                avg_values.append(np.nan)
                min_values.append(np.nan)
                max_values.append(np.nan)
                point_labels.append("")
            else:
                avg_values.append(row["average_runtime_seconds"].iloc[0])
                min_values.append(row["min_runtime_seconds"].iloc[0])
                max_values.append(row["max_runtime_seconds"].iloc[0])

                input_entities = row["input_entities"].iloc[0]
                matched = row["matched_entities"].iloc[0]
                returned = row["returned_results"].iloc[0]
                
                point_labels.append(f"({int(input_entities)},{int(matched)},{int(returned)})"
                )

        avg_values = np.array(avg_values, dtype=float)
        min_values = np.array(min_values, dtype=float)
        max_values = np.array(max_values, dtype=float)

        # shaded min-max range
        ax.fill_between(
            x_positions,
            min_values,
            max_values,
            alpha=0.20
        )

        # average runtime line
        ax.plot(
            x_positions,
            avg_values,
            marker="o",
            linewidth=2
        )

        # annotate each point: (input, matched, returned)
        for i, (x, y, label) in enumerate(zip(x_positions, avg_values, point_labels)):
            if pd.notna(y) and label:
                
                # default
                x_offset = 0
                y_offset = 6
                ha = "center"
                
                # first point: move label slightly right
                if i == 0:
                   x_offset = 0
                   ha = "left"
                
                # last point: move label slightly left
                elif i == len(x_positions) - 1:
                     x_offset = 0
                     ha = "right"

                ax.annotate(
                    label,
                    (x, y),
                    textcoords="offset points",
                    xytext=(x_offset, y_offset),
                    ha=ha,
                    va="bottom",
                    fontsize=7,
                    clip_on=True
                )

        ax.set_title(template_label_map.get(template, template), fontsize=10)
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels)

        # same y-axis range for all subplots
        ax.set_ylim(0, upper_limit)

        # fixed y tick interval
        ax.yaxis.set_major_locator(MultipleLocator(y_tick_step))
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

        # show y tick labels on EVERY subplot
        ax.tick_params(axis='y', labelleft=True)

        ax.grid(True, axis="y", linestyle="--", alpha=0.4)

#    main title
    fig.suptitle(
        f"Scalability results by competency questions and query template in the {domain_name} domain",
        fontsize=14
    )

    fig.supxlabel("Number of input entities")
    fig.supylabel("Average runtime (seconds)")
    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="-",
            linewidth=2,
            label="Average runtime"
        ),
        Patch(
            alpha=0.20,
            label="Min-max runtime range"
        )
    ]
    
    fig.legend(
        handles=legend_handles,
        title="Point annotations: (input entities, matched entities, returned results)",
        loc="upper right",
        bbox_to_anchor=(0.98, 0.98),
        frameon=False,
        fontsize=8,
        title_fontsize=8
    )

    plt.tight_layout(rect=[0.01, 0.01, 1, 0.94])
    
    output_file = f"{domain_name}_scalability_current.png"
    plt.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_file)
    print("Common y-axis upper limit:", upper_limit)
    print("Y tick step:", y_tick_step)


# 5. Draw figures

plot_small_plots_with_range("business", y_tick_step=0.2)
plot_small_plots_with_range("politician", y_tick_step=0.2)

## Reproduce thesis summary from the saved raw log

This cell reproduces the summary table used in the thesis. It reads the saved raw log from the original scalability evaluation:

`qlever_all_query_log_20260614_025034.csv`

The output files are the thesis result files:

`query_results_with_summary.xlsx`  
`query_results_with_summary.csv`

In [ ]:
import pandas as pd
import re

# 1. Read the raw log generated by the previous cell
input_file = "qlever_all_query_log_20260614_025034.csv"
df = pd.read_csv(input_file)

# 2. Parse domain, scale, cq from query_file
def parse_query_file(query_file):
    # example: business_scale10_CQ1_recorded_value.sparql
    name = str(query_file).replace(".sparql", "")
    
    domain_match = re.match(r"(business|politician)_", name)
    scale_match = re.search(r"scale(\d+)", name)
    cq_match = re.search(r"(CQ\d+)", name)
    
    domain = domain_match.group(1) if domain_match else None
    scale = int(scale_match.group(1)) if scale_match else None
    cq = cq_match.group(1) if cq_match else None
    
    # template name without domain and scale
    template = name
    template = re.sub(r"^(business|politician)_scale\d+_", "", template)
    
    return pd.Series({
        "domain": domain,
        "scale": scale,
        "cq": cq,
        "template": template
    })

parsed = df["query_file"].apply(parse_query_file)
df = pd.concat([parsed, df], axis=1)

# 3. Only use successful runs to calculate runtime
success_df = df[df["status"] == "success"].copy()

# 4. Pivot 5 runtimes horizontally：run_1, run_2, ..., run_5
runtime_wide = success_df.pivot_table(
    index=["domain", "cq", "template", "scale", "query_file"],
    columns="run_id",
    values="runtime_seconds",
    aggfunc="first"
).reset_index()

runtime_wide = runtime_wide.rename(columns={
    1: "run_1_runtime_seconds",
    2: "run_2_runtime_seconds",
    3: "run_3_runtime_seconds",
    4: "run_4_runtime_seconds",
    5: "run_5_runtime_seconds",
})

# 5. Summary basic info for each query template
summary_base = (
    df.groupby(["domain", "cq", "template", "scale", "query_file"], dropna=False)
    .agg(
        total_runs=("run_id", "count"),
        success_runs=("status", lambda x: (x == "success").sum()),
        timeout_runs=("status", lambda x: (x == "timeout").sum()),
        failed_runs=("status", lambda x: ((x != "success") & (x != "timeout")).sum()),
        input_entities=("input_entities", "max"),
        matched_entities=("matched_entities", "first"),
        returned_results=("result_rows", "first"),
    )
    .reset_index()
)

# 6. Summary runtime：average, min, max, range, standard deviation
runtime_summary = (
    success_df.groupby(["domain", "cq", "template", "scale", "query_file"], dropna=False)
    .agg(
        average_runtime_seconds=("runtime_seconds", "mean"),
        min_runtime_seconds=("runtime_seconds", "min"),
        max_runtime_seconds=("runtime_seconds", "max"),
        std_runtime_seconds=("runtime_seconds", "std"),
    )
    .reset_index()
)

summary = summary_base.merge(
    runtime_summary,
    on=["domain", "cq", "template", "scale", "query_file"],
    how="left"
)

summary["runtime_range_seconds"] = (
    summary["max_runtime_seconds"] - summary["min_runtime_seconds"]
)

summary["success_rate"] = summary["success_runs"] / summary["total_runs"]

# 7. Merge the 5 specific runtimes
summary = summary.merge(
    runtime_wide,
    on=["domain", "cq", "template", "scale", "query_file"],
    how="left"
)

# 8. Mark prominent cases
def mark_prominent(row):
    reasons = []

    if row["timeout_runs"] > 0:
        reasons.append("has timeout")

    if row["failed_runs"] > 0:
        reasons.append("has failed run")

    if pd.notna(row["runtime_range_seconds"]) and row["runtime_range_seconds"] > 0.5:
        reasons.append("large runtime variation")

    if pd.notna(row["average_runtime_seconds"]) and row["average_runtime_seconds"] > 1:
        reasons.append("slow query")

    if len(reasons) == 0:
        return "normal"
    else:
        return "; ".join(reasons)

summary["prominent_note"] = summary.apply(mark_prominent, axis=1)

# 9. Sort by domain, CQ, scale
# CQ1, scale 1, 10, 100, 1000; then CQ2, scale 1, 10, 100, 1000
cq_order = {f"CQ{i}": i for i in range(1, 11)}
scale_order = {1: 1, 10: 2, 100: 3, 1000: 4}
domain_order = {"business": 1, "politician": 2}

summary["domain_order"] = summary["domain"].map(domain_order)
summary["cq_order"] = summary["cq"].map(cq_order)
summary["scale_order"] = summary["scale"].map(scale_order)

summary = summary.sort_values(
    by=["domain_order", "cq_order", "template", "scale_order"]
)

summary = summary.drop(columns=["domain_order", "cq_order", "scale_order"])

# 10. Adjust column order 
columns_order = [
    "domain",
    "cq",
    "template",
    "scale",
    "query_file",
    "total_runs",
    "success_runs",
    "timeout_runs",
    "failed_runs",
    "success_rate",
    "input_entities",
    "matched_entities",
    "returned_results",
    "average_runtime_seconds",
    "min_runtime_seconds",
    "max_runtime_seconds",
    "runtime_range_seconds",
    "std_runtime_seconds",
    "prominent_note",
    "run_1_runtime_seconds",
    "run_2_runtime_seconds",
    "run_3_runtime_seconds",
    "run_4_runtime_seconds",
    "run_5_runtime_seconds",
]

summary = summary[[col for col in columns_order if col in summary.columns]]

# 11. Write to the second sheet
output_xlsx = "query_results_with_summary.xlsx"
output_csv = "query_results_with_summary.csv

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="all_runs_raw_log", index=False)
    summary.to_excel(writer, sheet_name="average_summary", index=False)

summary.to_csv(output_csv, index=False)

print(f"Input raw log: {input_file}")
print(f"Saved to {output_xlsx}")
print(f"Saved to {output_csv}")

## Reproduce thesis scalability figures

This cell reproduces the scalability figures reported in the thesis. It reads the thesis summary file:

`query_results_with_summary.xlsx`

The output figures are:

`business_scalability.png`  
`politician_scalability.png`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import math


file_path = "query_results_with_summary.xlsx"
df = pd.read_excel(file_path, sheet_name="average_summary")

# 2. Keep needed data
# Remove CQ4 because it shares the same retrieval structure as CQ1
# but mention CQ4 together with CQ1 in the subplot title
df = df[df["cq"] != "CQ4"].copy()

# Keep only business and politician
df = df[df["domain"].isin(["business", "politician"])].copy()


# 3. Define template order and labels

template_order = [
    "CQ1_recorded_value",
    "CQ2_current_value_qualifier",
    "CQ2_current_value_qualifier_rank",
    "CQ3_value_at_2024",
    "CQ5_CQ8_temporal_info",
    "CQ9_sequence_qualifier",
    "CQ9_sequence_property",
    "CQ10_reference",
]

template_label_map = {
    "CQ1_recorded_value": "CQ1, CQ4\nRecorded value retrieval",
    "CQ2_current_value_qualifier": "CQ2\nCurrent value retrieval\nwith qualifier",
    "CQ2_current_value_qualifier_rank": "CQ2\nCurrent value retrieval\nwith qualifier and rank",
    "CQ3_value_at_2024": "CQ3\nValue retrieval at a given time",
    "CQ5_CQ8_temporal_info": "CQ5–CQ8\nTemporal information retrieval",
    "CQ9_sequence_qualifier": "CQ9\nSequence relation retrieval\nwith qualifier",
    "CQ9_sequence_property": "CQ9\nSequence relation retrieval\nwith property",
    "CQ10_reference": "CQ10\nReference retrieval",
}

scale_order = [1, 10, 100, 1000]

df = df[df["template"].isin(template_order)].copy()
df["template_order"] = df["template"].map({t: i for i, t in enumerate(template_order)})
df["scale_order"] = df["scale"].map({s: i for i, s in enumerate(scale_order)})
df = df.sort_values(["domain", "template_order", "scale_order"])


# 4. Plot function

def plot_small_plots_with_range(domain_name, y_tick_step=0.2):
    domain_df = df[df["domain"] == domain_name].copy()

    # ---- determine one common y-axis range for all 8 subplots ----
    domain_ymax = domain_df["max_runtime_seconds"].max()

    # round up to nearest tick step
    upper_limit = math.ceil(domain_ymax / y_tick_step) * y_tick_step

    # give a little extra space for annotations
    upper_limit = upper_limit + y_tick_step

    fig, axes = plt.subplots(
        2, 4,
        figsize=(19, 8.1),
        sharex=True,
        sharey=True   # all subplots use the same y scale
    )

    axes = axes.flatten()

    x_positions = np.arange(len(scale_order))
    x_labels = [str(s) for s in scale_order]

    for ax, template in zip(axes, template_order):
        temp_df = domain_df[domain_df["template"] == template].copy()
        temp_df = temp_df.sort_values("scale_order")

        avg_values = []
        min_values = []
        max_values = []
        point_labels = []

        for scale in scale_order:
            row = temp_df[temp_df["scale"] == scale]

            if row.empty:
                avg_values.append(np.nan)
                min_values.append(np.nan)
                max_values.append(np.nan)
                point_labels.append("")
            else:
                avg_values.append(row["average_runtime_seconds"].iloc[0])
                min_values.append(row["min_runtime_seconds"].iloc[0])
                max_values.append(row["max_runtime_seconds"].iloc[0])

                input_entities = row["input_entities"].iloc[0]
                matched = row["matched_entities"].iloc[0]
                returned = row["returned_results"].iloc[0]
                
                point_labels.append(f"({int(input_entities)},{int(matched)},{int(returned)})"
                )

        avg_values = np.array(avg_values, dtype=float)
        min_values = np.array(min_values, dtype=float)
        max_values = np.array(max_values, dtype=float)

        # shaded min-max range
        ax.fill_between(
            x_positions,
            min_values,
            max_values,
            alpha=0.20
        )

        # average runtime line
        ax.plot(
            x_positions,
            avg_values,
            marker="o",
            linewidth=2
        )

        # annotate each point: (input, matched, returned)
        for i, (x, y, label) in enumerate(zip(x_positions, avg_values, point_labels)):
            if pd.notna(y) and label:
                
                # default
                x_offset = 0
                y_offset = 6
                ha = "center"
                
                # first point: move label slightly right
                if i == 0:
                   x_offset = 0
                   ha = "left"
                
                # last point: move label slightly left
                elif i == len(x_positions) - 1:
                     x_offset = 0
                     ha = "right"

                ax.annotate(
                    label,
                    (x, y),
                    textcoords="offset points",
                    xytext=(x_offset, y_offset),
                    ha=ha,
                    va="bottom",
                    fontsize=7,
                    clip_on=True
                )

        ax.set_title(template_label_map.get(template, template), fontsize=10)
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels)

        # same y-axis range for all subplots
        ax.set_ylim(0, upper_limit)

        # fixed y tick interval
        ax.yaxis.set_major_locator(MultipleLocator(y_tick_step))
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

        # show y tick labels on EVERY subplot
        ax.tick_params(axis='y', labelleft=True)

        ax.grid(True, axis="y", linestyle="--", alpha=0.4)

#    main title
    fig.suptitle(
        f"Scalability results by competency questions and query template in the {domain_name} domain",
        fontsize=14
    )

    fig.supxlabel("Number of input entities")
    fig.supylabel("Average runtime (seconds)")
    legend_handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="-",
            linewidth=2,
            label="Average runtime"
        ),
        Patch(
            alpha=0.20,
            label="Min-max runtime range"
        )
    ]
    
    fig.legend(
        handles=legend_handles,
        title="Point annotations: (input entities, matched entities, returned results)",
        loc="upper right",
        bbox_to_anchor=(0.98, 0.98),
        frameon=False,
        fontsize=8,
        title_fontsize=8
    )

    plt.tight_layout(rect=[0.01, 0.01, 1, 0.94])
    
    output_file = f"{domain_name}_scalability.png"
    plt.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_file)
    print("Common y-axis upper limit:", upper_limit)
    print("Y tick step:", y_tick_step)


# 5. Draw figures

plot_small_plots_with_range("business", y_tick_step=0.2)
plot_small_plots_with_range("politician", y_tick_step=0.2)